In [10]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [11]:
# CONFIG
MODEL_DIR = r"D:/LPA_MTech_Project/My_Models/LegalBERT_Models/legalbert_uncased_1990-2025_medium_1/best_model"
PARQUET_PATH = "D:/LPA_MTech_Project/Enriched_Datasets/SC_1990_2025_enriched_IPC.parquet"
QUERY_CASE_PATH = "2020_1_57_68_EN.txt"

TOP_K = 10
MAX_LENGTH = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEXT_COLS = [
    "facts_section",
    "issues_section",
    "reasoning_section"
]

In [12]:
# LOAD MODEL
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModel.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [13]:
# EMBEDDING HELPERS
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1)

def embed_text(text: str):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        output = model(**inputs)

    # return mean_pooling(output, inputs["attention_mask"]).cpu().numpy()
    emb = mean_pooling(output, inputs["attention_mask"])
    return emb.squeeze(0).cpu().numpy()

In [14]:
# LOAD DATA
df = pd.read_parquet(PARQUET_PATH)

with open(QUERY_CASE_PATH, "r", encoding="utf-8") as f:
    query_text = f.read()

In [15]:
# BASIC FILTER 
candidates = df[
    (df["has_ipc"] == True) |
    (df["num_unique_acts"] > 0)
].copy()

In [16]:
# BUILD EMBEDDING TEXT
def build_case_text(row):
    parts = []
    for col in TEXT_COLS:
        if pd.notna(row[col]):
            parts.append(row[col])
    return " ".join(parts)

candidates["embed_text"] = candidates.apply(build_case_text, axis=1)
candidates = candidates[candidates["embed_text"].str.len() > 50]

In [17]:
# EMBEDDINGS
query_emb = embed_text(query_text)

# doc_embs = []
# for text in candidates["embed_text"]:
#     doc_embs.append(embed_text(text))

doc_embs = np.vstack([
    embed_text(text) for text in candidates["embed_text"]
])

In [18]:
# SIMILARITY
# scores = cosine_similarity(query_emb, doc_embs)[0]
scores = cosine_similarity(
    query_emb.reshape(1, -1),
    doc_embs
)[0]
candidates["similarity"] = scores

In [19]:
print(query_emb.shape)   # (768,)
print(doc_embs.shape)    # (N, 768)


(768,)
(14265, 768)


In [20]:
# TOP-K PRECEDENTS
topk = candidates.sort_values(
    "similarity", ascending=False
).head(TOP_K)

In [21]:
# OUTPUT
print("\nTop Supreme Court Precedents:\n")

for _, row in topk.iterrows():
    print(
        f"Case ID: {row['case_id']} | Year: {row['year']}\n"
        f"Bench: {row['bench_size']} | Judges: {row['judges']}\n"
        f"IPC: {row['ipc_sections']} | Acts: {row['act_names']}\n"
        f"Verdict: {row['verdict_label']} | Similarity: {row['similarity']:.3f}\n"
        f"{'-'*80}"
    )


Top Supreme Court Precedents:

Case ID: 2024_3_1228_1248 | Year: 2024
Bench: 1 | Judges: ['BHUSHAN RAMKRISHNA GAVAI']
IPC: [] | Acts: ['Protection of Children from Sexual Offences Act, 2012'
 'Juvenile Justice Act, 1986']
Verdict: 1.0 | Similarity: 0.983
--------------------------------------------------------------------------------
Case ID: S_2006_2_765_771 | Year: 2006
Bench: 2 | Judges: ['AR. LAKSHMANAN' 'LOKESHWAR SINGH PANTA']
IPC: [] | Acts: ['Dowry Prohibition Act, 1961']
Verdict: 0.0 | Similarity: 0.983
--------------------------------------------------------------------------------
Case ID: 2019_6_1158_1171 | Year: 2019
Bench: 1 | Judges: ['ASHOK BHUSHAN']
IPC: [] | Acts: ['Lokpal and Lokayuktas Act, 2013']
Verdict: 1.0 | Similarity: 0.983
--------------------------------------------------------------------------------
Case ID: 2007_5_221_232 | Year: 2007
Bench: 1 | Judges: ['S.B. SINHA']
IPC: [] | Acts: ['Arms Act, 1959' 'Evidence Act, 1872']
Verdict: 1.0 | Similarity: 0.98